# M3.F1 Chatbot Lab: Train a LSTM-Based Model
## Author: Jacob Mendez

### Imports

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

### Loading Corpus

In [ ]:
def load_docs(files):
    text = ""
    for file in files:
        with open(file, "r", encoding="utf-8") as f:
            text += f.read().lower() + " "
    return text

files = ["MTG_Rules.txt"]
text = load_docs(files)



### Tokenization

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1


### Sequencing Text

In [ ]:
sequence_length = 20
input_sequences = []

for line in text.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(sequence_length, len(token_list)):
        n_gram = token_list[:i+1]
        input_sequences.append(token_list[i-sequence_length:i+1])

max_sequence_length = max(len(seq) for seq in input_sequences)

input_sequences = np.array(input_sequences)

X = input_sequences[:, :-1]
Y = input_sequences[:, -1]

print(f"X Head: {X}")
print(f"Y Head: {Y}")

### Modeling

#### Model 1: Small Size

In [ ]:
model = Sequential([
    Embedding(total_words, 64, input_length=max_sequence_length-1),
    LSTM(128),
    Dense(total_words, activation='softmax')
])

#### Model 2: Medium Size

In [ ]:
model = Sequential([
    Embedding(total_words, 128, input_length=max_sequence_length-1),
    LSTM(256, return_sequences=True),
    LSTM(256),
    Dense(total_words, activation='softmax')
])

#### Model 3: Large Size

In [ ]:
model = Sequential([
    Embedding(total_words, 256, input_length=max_sequence_length-1),
    LSTM(512, return_sequences=True),
    LSTM(512),
    Dense(total_words, activation='softmax')
])

### Compile Model

In [ ]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam'
)

### Training

In [ ]:
history = model.fit(X, Y, epochs=50, verbose=1) # 30 small, 50 medium, 75 large

### Generating Text

In [ ]:
def generate_text(seed_text, next_words=100):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_length-1,
            padding='pre'
        )

        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break

        seed_text += " " + output_word

    return seed_text


print(generate_text("If a creature dies", 80))